# 03 — Hipótesis longitudinal con datos reales (EMBED)

**El flujo central del TFG (O7/O8), ejecutable de principio a fin.** Para cada
paciente comparamos su exploración previa "normal" (t-1) con una exploración
posterior (t): en los pacientes *incidentes* la posterior es donde se
diagnostica la sospecha/malignidad; en los *control*, todas son normales.
Medimos dos señales topológicas:

- **Anticipación**: descriptores de persistencia de t-1 **sola** — ¿ya había señal *antes* del diagnóstico?
- **Deriva**: distancia *bottleneck* entre los diagramas de t-1 y t — ¿cuánto cambió la topología?

Reutiliza exactamente el mismo código que `scripts/download_embed_longitudinal.py`
y `scripts/embed_longitudinal_evaluation.py` (este notebook es la vista
interactiva; los scripts, la reproducción de un tirón). La parte de
clasificación aislada sobre CBIS-DDSM vive en `02_real_data_pipeline.ipynb`.

## Requisitos previos (una sola vez)

1. **Acceso a EMBED concedido** (es de acceso controlado; pasos en
   `docs/acceso_datasets_longitudinales.pdf`).
2. **Credenciales AWS** en `~/.aws/credentials`, perfil `[emory-embed]`.
   *Nunca* pongas las claves en el notebook ni en ningún fichero del repo.
3. **Tablas de metadatos** descargadas en `data/embed/` (celda 1 lo hace si faltan).
4. Dependencias: `pip install -r requirements.txt` (incluye `boto3` y
   `pylibjpeg`, necesario porque los DICOM de EMBED vienen comprimidos en
   JPEG Lossless).


## 0. Imports y configuración

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "scripts"))

from src.topology import backend
import download_embed_longitudinal as dl
import embed_longitudinal_evaluation as ev

EMBED_DIR = ROOT / "data" / "embed"
print("Backend TDA activo:", backend())


## 1. Tablas de metadatos clínicos

Dos CSV pequeños (frente a los TB de imágenes): `clinical.csv` tiene una fila
por hallazgo con el BI-RADS (`asses`) de cada exploración, y
`metadata_reduced.csv` la ruta S3 de cada DICOM. Esta celda los descarga solo
si no están ya en `data/embed/`.

In [ ]:
import boto3

EMBED_DIR.mkdir(parents=True, exist_ok=True)
TABLAS = {
    "clinical.csv": "tables/EMBED_OpenData_clinical.csv",
    "metadata_reduced.csv": "tables/EMBED_OpenData_metadata_reduced.csv",
}
faltan = {local: remota for local, remota in TABLAS.items() if not (EMBED_DIR / local).exists()}
if faltan:
    s3 = boto3.Session(profile_name="emory-embed", region_name="us-west-2").client("s3")
    for local, remota in faltan.items():
        print(f"Descargando {remota}...")
        s3.download_file("embed-dataset-open", remota, str(EMBED_DIR / local))
else:
    print("Tablas ya descargadas.")


## 2. Estructura longitudinal de la cohorte

¿Cuántos pacientes tienen 2+ exploraciones? ¿Cuántos son candidatos
*incidentes* (primera exploración Normal/Benigna → posterior
Sospechosa/Maligna) y cuántos *control*? Reutiliza
`dl._load_per_study_assessment` y `dl.select_pairs`.

In [ ]:
per_study = dl._load_per_study_assessment(EMBED_DIR / "clinical.csv")

exams = per_study.groupby("empi_anon").size()
print(f"{per_study['empi_anon'].nunique()} pacientes, {len(per_study)} exploraciones")
print(f"{(exams >= 2).sum()} pacientes con 2+ exploraciones")

fig, ax = plt.subplots(figsize=(6, 3))
exams.value_counts().sort_index().plot.bar(ax=ax, color="#1d5a99")
ax.set_xlabel("exploraciones por paciente"); ax.set_ylabel("pacientes")
ax.set_title("Estructura longitudinal de EMBED")
plt.tight_layout(); plt.show()


## 3. Diseño del estudio: caso-control emparejado por intervalo

**Incidentes**: pacientes cuya primera exploración es Normal/Benigna
(BI-RADS `N`/`B`) y una posterior es Sospechosa/Maligna (`S`/`M`/`K`); par
(t-1, t) = (primera normal, primera maligna). `dl.select_pairs`, semilla fija.

**Controles**: para **cada** incidente se busca, en el pool completo de
pacientes con todas sus exploraciones Normal/Benigna, uno cuyo intervalo
t-1 → t esté a ±90 días del intervalo del incidente
(`dl.select_interval_matched_controls`). Este emparejamiento no es un
detalle: sin él, el tiempo transcurrido y la época de adquisición
distinguen las clases por sí solos (ver H6, sección 14).

In [ ]:
pares = dl.select_pairs(per_study, n_incident=200, n_control=200, seed=42)
print(f"{sum(p['kind'] == 'incidente' for p in pares)} incidentes seleccionados "
      f"(semilla 42); los controles se emparejan a ellos por intervalo")


## 4. Descarga del piloto (reanudable)

En ambas exploraciones se usa **la misma vista** (lateralidad + proyección,
solo tipo `2D`), para comparar la misma región anatómica. Los ficheros ya
descargados se saltan. (~8 MB por paciente.)

Por terminal, en dos pasos:
```
python scripts/download_embed_longitudinal.py --n-incident 200 --n-control 200   # incidentes (+ controles del diseño original)
python scripts/download_embed_longitudinal.py --match-interval                   # controles emparejados 1:1
```

In [ ]:
MANIFIESTO = "longitudinal_pilot_manifest_matched.csv"   # diseño emparejado (el correcto)
MANIFIESTO_ORIGINAL = "longitudinal_pilot_manifest.csv"  # solo para la comparación de la H6

manifest = pd.read_csv(EMBED_DIR / MANIFIESTO)
print(f"{len(manifest)} pacientes: {(manifest['kind'] == 'incidente').sum()} incidentes, "
      f"{(manifest['kind'] == 'control').sum()} controles emparejados")
manifest.head()


## 5. Un par real (t-1, t) de cada clase

La mejor intuición de qué estamos midiendo: en el incidente, la exploración
t-1 fue leída como normal y en t hay un hallazgo; en el control, ambas son
normales.

In [ ]:
def _primer(kind):
    fila = manifest[manifest["kind"] == kind].iloc[0]
    return fila, ev._load_and_preprocess(ROOT / fila["path_t0"]), ev._load_and_preprocess(ROOT / fila["path_t1"])

fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for i, kind in enumerate(["incidente", "control"]):
    fila, img_t0, img_t1 = _primer(kind)
    axes[i, 0].imshow(img_t0, cmap="gray"); axes[i, 0].set_title(f"{kind} {fila['empi_anon']}\n t-1 (normal)")
    axes[i, 1].imshow(img_t1, cmap="gray"); axes[i, 1].set_title("t (posterior)")
    for ax in axes[i]:
        ax.axis("off")
plt.tight_layout(); plt.show()


## 6. Señales topológicas de todo el piloto

Para cada paciente: diagrama de persistencia de t-1 y de t → vector de
anticipación (Persistence Image de t-1) y deriva (distancia bottleneck H0/H1
entre ambos, con `max_points=100`: los diagramas reales tienen miles de
puntos y sin recortar la distancia tarda ~2 min/par en vez de ~0.2 s;
recortar al top-100 por persistencia es la práctica estándar).

⏱️ La primera vez ~1 s/paciente; las features se guardan en
`data/embed/features_cache.pkl` y las siguientes ejecuciones son instantáneas.

In [ ]:
X_anticipacion, X_deriva, y = ev.compute_features(manifest, n_bins=6, verbose=False)
X_combinada = np.hstack([X_anticipacion, X_deriva])
print("anticipacion:", X_anticipacion.shape, " deriva:", X_deriva.shape)


## 7. ¿Separan las señales a incidentes de controles?

AUC por validación cruzada (5-fold) con Random Forest, igual que
`scripts/embed_longitudinal_evaluation.py`. Con ~100 pacientes esto sigue
siendo un **piloto**: mira las desviaciones típicas antes de sacar
conclusiones.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

resultados = {}
for nombre, X in [("anticipación", X_anticipacion), ("deriva", X_deriva), ("combinada", X_combinada)]:
    s = cross_val_score(RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced"),
                        X, y, cv=5, scoring="roc_auc")
    resultados[nombre] = (s.mean(), s.std())
    print(f"{nombre:12s}: AUC = {s.mean():.3f} +/- {s.std():.3f}")

fig, ax = plt.subplots(figsize=(5, 3))
nombres = list(resultados)
medias = [resultados[n][0] for n in nombres]
stds = [resultados[n][1] for n in nombres]
ax.bar(nombres, medias, yerr=stds, capsize=4, color=["#888888", "#1d5a99", "#5a991d"])
ax.axhline(0.5, color="k", ls="--", lw=0.8, label="azar")
ax.set_ylabel("AUC (5-fold CV)"); ax.set_ylim(0, 1); ax.legend()
plt.tight_layout(); plt.show()


## 8. La distribución de la deriva, a ojo

Si la deriva separa, los incidentes deberían tener distancias t-1→t mayores
que los controles.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
for etiqueta, color in [(1, "#b03030"), (0, "#1d5a99")]:
    vals = X_deriva[y == etiqueta, 1]  # distancia H1
    ax.hist(vals, bins=15, alpha=0.6, color=color,
            label="incidente" if etiqueta else "control")
ax.set_xlabel("distancia bottleneck H1 (t-1 → t)"); ax.set_ylabel("pacientes"); ax.legend()
plt.tight_layout(); plt.show()


## 9. Hipótesis secundarias: covariables clínicas por paciente

El AUC global no es el final de la historia. EMBED trae covariables que
permiten **más casos de estudio**: edad (`age_at_study`), densidad mamaria
BI-RADS (`tissueden`) y las fechas de ambas exploraciones (el intervalo
t-1 → t). El manifiesto guarda el `acc_anon` de ambas exploraciones, así que
basta con cruzarlo con `clinical.csv`.

In [ ]:
cov = manifest[["empi_anon", "kind", "acc_t0", "acc_t1"]].copy()
cov["intervalo_dias"] = ev.intervals_days(manifest)

clin = pd.read_csv(EMBED_DIR / "clinical.csv", usecols=["empi_anon", "acc_anon", "age_at_study", "tissueden"])
clin = clin.drop_duplicates(["empi_anon", "acc_anon"]).set_index(["empi_anon", "acc_anon"])
claves_t0 = list(zip(cov["empi_anon"], cov["acc_t0"]))
cov["edad_t0"] = clin["age_at_study"].reindex(claves_t0).values
cov["densidad_t0"] = clin["tissueden"].reindex(claves_t0).values
cov[["empi_anon", "kind", "intervalo_dias", "edad_t0", "densidad_t0"]].head()


## 10. H2 — ¿Es la edad un factor de confusión?

Si los incidentes fueran sistemáticamente mayores que los controles, parte
de cualquier señal podría ser "edad" y no "topología". Comparamos las
distribuciones (Mann-Whitney) y reajustamos la deriva por edad con una
regresión logística.

In [ ]:
from scipy import stats
import statsmodels.api as sm

ok = cov["edad_t0"].notna().values
edad_inc = cov.loc[ok & (y == 1), "edad_t0"]
edad_ctl = cov.loc[ok & (y == 0), "edad_t0"]
u, p_edad = stats.mannwhitneyu(edad_inc, edad_ctl)
print(f"Edad en t-1: incidentes {edad_inc.mean():.1f} ± {edad_inc.std():.1f} vs "
      f"controles {edad_ctl.mean():.1f} ± {edad_ctl.std():.1f}  (Mann-Whitney p={p_edad:.4f})")

X_log = sm.add_constant(pd.DataFrame({
    "deriva_h0": X_deriva[ok, 0],
    "deriva_h1": X_deriva[ok, 1],
    "edad": cov.loc[ok, "edad_t0"].values,
}).astype(float))
modelo = sm.Logit(y[ok], X_log).fit(disp=0)
print(modelo.summary2().tables[1].round(4))
print()
print("Si los coeficientes de deriva siguen siendo significativos con la edad en el")
print("modelo, la señal de deriva no se explica solo por la edad.")


## 11. H3 — ¿Depende la señal de la densidad mamaria?

La densidad alta (BI-RADS C/D) dificulta la lectura radiológica. ¿También
degrada la señal topológica de deriva? AUC por subgrupo de densidad
(donde haya n suficiente).

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

print("AUC de deriva por densidad mamaria (solo subgrupos con >=15 pacientes y ambas clases):")
for dens, grupo in cov.groupby("densidad_t0"):
    idx = grupo.index.values
    y_g = y[idx]
    if len(idx) < 15 or len(np.unique(y_g)) < 2:
        print(f"  densidad {dens}: n={len(idx)} (insuficiente)")
        continue
    s = cross_val_score(RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced"),
                        X_deriva[idx], y_g, cv=3, scoring="roc_auc")
    print(f"  densidad {dens}: n={len(idx)}  AUC={s.mean():.3f} +/- {s.std():.3f}")


## 12. H4 — ¿Más deriva cuanto más tiempo pasa? ¿Y la anticipación, cuanto más cerca del diagnóstico?

Dos preguntas temporales:
- En **controles**, la deriva debería crecer suavemente con el intervalo
  (envejecimiento del tejido). En **incidentes**, debería estar dominada por
  la lesión, no por el tiempo.
- La pregunta clínica de oro: en incidentes, ¿la exploración t-1 más
  **cercana** al diagnóstico muestra más señal de anticipación?

In [ ]:
for etiqueta, nombre in [(0, "controles"), (1, "incidentes")]:
    idx = (y == etiqueta) & cov["intervalo_dias"].notna().values
    rho, p = stats.spearmanr(cov.loc[idx, "intervalo_dias"], X_deriva[idx, 1])
    print(f"deriva H1 vs intervalo ({nombre}): Spearman rho={rho:+.3f} (p={p:.4f}, n={idx.sum()})")

# anticipación por cercanía al diagnóstico (solo incidentes)
inc = (y == 1) & cov["intervalo_dias"].notna().values
mediana = cov.loc[inc, "intervalo_dias"].median()
cerca = inc & (cov["intervalo_dias"] <= mediana).values
lejos = inc & (cov["intervalo_dias"] > mediana).values
ctrl = (y == 0)

from sklearn.model_selection import cross_val_score
for nombre, mask in [("t-1 CERCA del diagnostico", cerca), ("t-1 LEJOS del diagnostico", lejos)]:
    idx = np.where(mask | ctrl)[0]
    s = cross_val_score(RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced"),
                        X_anticipacion[idx], y[idx], cv=5, scoring="roc_auc")
    print(f"anticipacion, {nombre} (mediana={mediana:.0f} dias): AUC={s.mean():.3f} +/- {s.std():.3f} (n_inc={mask.sum()})")


## 13. H5 — ¿Qué dimensión de homología lleva la señal: H0 o H1?

La deriva tiene dos componentes: distancia bottleneck en H0 (componentes
conexas) y en H1 (ciclos/agujeros). ¿Cuál separa mejor?

In [ ]:
for nombre, col in [("solo H0", [0]), ("solo H1", [1]), ("H0+H1", [0, 1])]:
    s = cross_val_score(RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced"),
                        X_deriva[:, col], y, cv=5, scoring="roc_auc")
    print(f"deriva {nombre:8s}: AUC={s.mean():.3f} +/- {s.std():.3f}")

u, p = stats.mannwhitneyu(X_deriva[y == 1, 1], X_deriva[y == 0, 1])
print(f"\nMann-Whitney deriva H1 incidentes vs controles: p={p:.2e}")


## 14. H6 — Dos confusores de diseño (caso de estudio metodológico)

Las secciones anteriores usan el diseño **emparejado**. Esta sección muestra
por qué hacía falta, comparándolo con el diseño **original** que se usó al
principio: los controles con sus dos *primeras* exploraciones (seguimiento
corto) frente a incidentes con seguimientos de años.

Dos confusores en el diseño original:
1. **Intervalo t-1 → t**: la deriva crece con el tiempo en ambos grupos
   (H4); si los incidentes tienen intervalos más largos, el tiempo hace
   parte del trabajo.
2. **Época de adquisición de t-1**: si los controles tienen su primera
   exploración en años distintos, cambios de equipo/procesado entre años
   pueden separar las clases mirando solo t-1 — es decir, contaminar la
   señal de anticipación.

In [ ]:
from scipy import stats

fechas_estudio = per_study.set_index(["empi_anon", "acc_anon"])["study_date"]

def _diseño(nombre):
    m = pd.read_csv(EMBED_DIR / nombre)
    Xa, Xd, yy = ev.compute_features(m, n_bins=6, verbose=False)
    dd = ev.intervals_days(m)
    anio = np.array([fechas_estudio[(r.empi_anon, r.acc_t0)].year for r in m.itertuples()])
    return dict(Xa=Xa, Xd=Xd, y=yy, dias=dd, anio=anio)

disenos = {"original": _diseño(MANIFIESTO_ORIGINAL), "emparejado": _diseño(MANIFIESTO)}
for nombre, d in disenos.items():
    yy = d["y"]
    p_int = stats.mannwhitneyu(d["dias"][yy == 1], d["dias"][yy == 0]).pvalue
    p_anio = stats.mannwhitneyu(d["anio"][yy == 1], d["anio"][yy == 0]).pvalue
    print(f"{nombre:11s}: intervalo mediana inc {np.median(d['dias'][yy == 1]):.0f} d vs ctl "
          f"{np.median(d['dias'][yy == 0]):.0f} d (p={p_int:.1e});  año de t-1 inc "
          f"{np.median(d['anio'][yy == 1]):.0f} vs ctl {np.median(d['anio'][yy == 0]):.0f} (p={p_anio:.1e})")

fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
for ax, (nombre, d) in zip(axes, disenos.items()):
    ax.hist(d["dias"][d["y"] == 1], bins=25, alpha=0.6, color="#b03030", label="incidente")
    ax.hist(d["dias"][d["y"] == 0], bins=25, alpha=0.6, color="#1d5a99", label="control")
    ax.set_title(f"diseño {nombre}"); ax.set_xlabel("intervalo t-1 → t (días)")
axes[0].set_ylabel("pacientes"); axes[0].legend()
plt.tight_layout(); plt.show()


### Las señales en ambos diseños

Si una señal se mantiene al pasar del diseño original al emparejado, es
topología; si se desploma, era un confusor. "Intervalo solo" es la línea
base: un AUC por encima de 0.5 con él significa que el diseño deja que el
tiempo distinga las clases sin mirar ninguna imagen.

In [ ]:
rf = lambda: RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced")
señales = ["intervalo solo", "anticipación", "deriva", "combinada"]
comparacion = {}
for nombre, d in disenos.items():
    for señal, X in [("intervalo solo", d["dias"][:, None]), ("anticipación", d["Xa"]),
                     ("deriva", d["Xd"]), ("combinada", np.hstack([d["Xa"], d["Xd"]]))]:
        s = cross_val_score(rf(), X, d["y"], cv=5, scoring="roc_auc")
        comparacion[(señal, nombre)] = (s.mean(), s.std())

tabla = pd.DataFrame({n: [f"{comparacion[(s, n)][0]:.3f} ± {comparacion[(s, n)][1]:.3f}" for s in señales]
                      for n in disenos}, index=señales)
display(tabla)

fig, ax = plt.subplots(figsize=(7, 3.2))
x = np.arange(len(señales)); w = 0.38
for k, (n, color) in enumerate([("original", "#999999"), ("emparejado", "#1d5a99")]):
    ax.bar(x + (k - 0.5) * w, [comparacion[(s, n)][0] for s in señales], w,
           yerr=[comparacion[(s, n)][1] for s in señales], capsize=3, color=color, label=n)
ax.axhline(0.5, color="k", ls="--", lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(señales); ax.set_ylim(0, 1); ax.set_ylabel("AUC (5-fold CV)")
ax.legend(); plt.tight_layout(); plt.show()


**Lectura**: con el diseño emparejado el "intervalo solo" cae al azar (el
confusor desaparece). La **deriva se sostiene**: es señal topológica real.
La **anticipación cae al azar**: la señal que aparentaba en el diseño
original venía de la época de adquisición de t-1, no de un precursor
topológico.

## 15. Casos de estudio extremos

El incidente con más deriva (la señal en su versión más clara) y el control
con más deriva (el "falso positivo" más peligroso del piloto: tejido que
cambió mucho sin malignidad).

In [ ]:
extremos = [
    ("incidente con MAS deriva", int(np.where(y == 1)[0][np.argmax(X_deriva[y == 1, 1])])),
    ("control con MAS deriva",   int(np.where(y == 0)[0][np.argmax(X_deriva[y == 0, 1])])),
]
fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for i, (titulo, idx) in enumerate(extremos):
    fila = manifest.iloc[idx]
    axes[i, 0].imshow(ev._load_and_preprocess(ROOT / fila["path_t0"]), cmap="gray")
    axes[i, 0].set_title(f"{titulo}\n{fila['empi_anon']} — t-1")
    axes[i, 1].imshow(ev._load_and_preprocess(ROOT / fila["path_t1"]), cmap="gray")
    axes[i, 1].set_title(f"t  (deriva H1 = {X_deriva[idx, 1]:.3f})")
    for ax in axes[i]:
        ax.axis("off")
plt.tight_layout(); plt.show()


## Lectura de los resultados y siguientes pasos

Resultados de referencia (diseño emparejado, 194 incidentes + 194 controles;
`results/embed_longitudinal_pilot_matched.txt`):

- **Deriva ≈ 0.75**: el cambio topológico entre la exploración previa y la
  del diagnóstico es una señal real, que sobrevive al control del intervalo
  y de la época de adquisición y que no explica la edad (H2). Va sobre todo
  por H0 (H5).
- **Anticipación ≈ 0.49**: al azar. Con este descriptor (imagen de
  persistencia de la vista completa a 224×224) **no se detecta un precursor
  topológico en la exploración leída como normal**. Es un resultado
  negativo, pero limpio: el aparente 0.63 del diseño original era un
  artefacto de época.

Siguientes pasos razonables:
- Descriptores más locales para la anticipación (recorte alrededor de la
  zona donde aparecerá la lesión, mayor resolución, filtración por
  transformada de distancia).
- Ampliar la cohorte emparejada (hay 644 incidentes candidatos).
